# Índice vectorial en GPU (Kaggle) — CATSNebula

Variante de `colab_encode_index.ipynb` para Kaggle. Mismo script, mismos
gates; lo que cambia es de dónde entra `chunks.jsonl` y dónde queda la salida:
Kaggle no monta Drive, usa **datasets** de entrada y `/kaggle/working` de
salida.

**Por qué Kaggle antes que Colab:** la sesión dura hasta 12 h garantizadas
(Colab gratis corta cerca de las 4 h y desconecta por inactividad) y la
corrida son 1-4 h. La cuota es de 30 h de GPU por semana, de sobra.

## Preparación (una vez, en la web de Kaggle)

1. **Verificar el teléfono** en `Settings → Phone Verification`. Sin eso no se
   puede activar internet, y sin internet no hay `pip install` ni descarga del
   modelo.
2. **Subir el corpus como dataset**: `Datasets → New Dataset`, subir
   `data/chunks.jsonl` (234 MB), título `catsnebula-chunks`, privado. Anotar el
   slug que queda en la URL.
3. En este notebook, panel derecho:
   - `Input → Add Input` → el dataset de arriba.
   - `Session options → Accelerator` → **GPU P100** (o T4 x2; el script usa una
     sola tarjeta).
   - `Session options → Internet` → **On**.

## Cómo correrlo

Interactivo hasta la celda 6 (muestra + verificación: son minutos), y la
corrida completa con **`Save Version → Save & Run All (Commit)`**, que la
ejecuta headless hasta 12 h y **conserva la salida** aunque cierres el
navegador. Una sesión interactiva que se cae se lleva `/kaggle/working` con
ella; una versión commiteada no.

Ojo: el commit re-ejecuta el notebook entero desde cero, así que la muestra de
la celda 5 vuelve a correr. Son unos minutos y revalida el mapeo 1:1 sobre la
misma GPU que construye el índice final, así que lo dejo a propósito.

In [ ]:
# 1. GPU e internet. Si falla el segundo, falta 'Internet: On' en el panel.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!curl -s -o /dev/null -w 'internet: %{http_code}\n' https://huggingface.co

In [ ]:
# 2. Dependencias. torch ya viene en la imagen de Kaggle compilado contra su
#    CUDA: pinnearlo rompe más de lo que arregla. El manifiesto registra cuál
#    se usó, que es lo que hace falta para reproducir la corrida.
!pip install -q sentence-transformers==5.7.0 transformers==5.15.0 faiss-cpu==1.15.0
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 3. Código del repo. Si el trabajo todavía está en una rama, agregar -b <rama>.
!git clone -q https://github.com/Juancabel/AdAstra2026-CATSNebula.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

In [ ]:
# 4. Entrada y salida.
#
#    /kaggle/input es de SOLO LECTURA: la salida va a /kaggle/working, que es
#    lo que Kaggle conserva como output de la versión commiteada.
from pathlib import Path

entradas = sorted(Path('/kaggle/input').glob('*/chunks.jsonl'))
assert entradas, 'No hay ningún dataset con chunks.jsonl. Ver "Add Input" arriba.'
CHUNKS = entradas[0]
SALIDA = Path('/kaggle/working/encoder_bge-m3')

print(CHUNKS, f'{CHUNKS.stat().st_size / 1e6:.0f} MB')
print(sum(1 for _ in CHUNKS.open('rb')), 'chunks')  # esperado: 86046

In [ ]:
# 5. Muestra de validación: mide ms/chunk en ESTA GPU y proyecta el total.
#    Descarga BGE-M3 (~2,3 GB) la primera vez.
!python encode_index.py "{CHUNKS}" /kaggle/working/muestra --muestra 500 --dispositivo cuda

In [ ]:
# 6. Mapeo 1:1 sobre la muestra, en un proceso nuevo. Si esto falla, NO seguir:
#    un índice desalineado recupera con puntajes razonables y devuelve el texto
#    equivocado, y ninguna métrica de recall lo delata.
!python scripts/verificar_indice.py /kaggle/working/muestra --n 30 --dispositivo cuda

In [ ]:
# 7. Corrida completa. Mirar antes la proyección de la celda 5.
#
#    Con 12 h de sesión y 1-4 h de trabajo no debería hacer falta reanudar. Si
#    hiciera falta: commitear la versión, añadir el output de esa versión como
#    input de la siguiente, copiar su carpeta .parcial/ a SALIDA y añadir
#    --reanudar acá. El script exige que batch_size, dispositivo y el corpus
#    sean los mismos; si no, se niega en vez de mezclar vectores incomparables.
!python encode_index.py "{CHUNKS}" "{SALIDA}" --dispositivo cuda

In [ ]:
# 8. Verificación final sobre el índice completo
!python scripts/verificar_indice.py "{SALIDA}" --n 50 --dispositivo cuda

In [ ]:
# 9. Entrega. Tras 'Save & Run All', los archivos quedan en la pestaña Output
#    de la versión, y de ahí se bajan uno por uno (index.faiss ~336 MB,
#    metadata.jsonl ~250 MB). No los comprimo: index.faiss son floats y no
#    comprime, y un zip de 600 MB es peor de bajar que dos archivos.
#
#    El clone del repo también vive en /kaggle/working y ensucia el output; se
#    borra para que queden solo los artefactos.
import json, shutil

shutil.rmtree('/kaggle/working/repo/.git', ignore_errors=True)
for f in sorted(SALIDA.iterdir()):
    print(f'{f.stat().st_size / 1e6:9.1f} MB  {f.name}')
print()
print(json.dumps(json.loads((SALIDA / 'manifiesto.json').read_text()), indent=2, ensure_ascii=False))